# Validating MarkovConsumerType TM Methods

This notebook validates the new transition-matrix methods added to
`MarkovConsumerType`:

- `define_distribution_grid()`
- `calc_transition_matrix()`
- `calc_ergodic_dist()`
- `compute_pe_steady_state()`

We verify these methods produce correct results by:
1. Checking column sums = 1 for a 2-state Markov model
2. Comparing `compute_pe_steady_state()` results with hand-built TM code
3. Verifying ergodic Markov state fractions match the analytical stationary distribution


In [1]:
import numpy as np
from copy import deepcopy
from HARK.ConsumptionSaving.ConsMarkovModel import (
    MarkovConsumerType,
    init_indshk_markov,
)

COLOR_MC = "tab:blue"
COLOR_TM = "tab:orange"

## 1. Create a 2-state symmetric Markov agent

Calibration is based on the default `init_indshk_markov` parameter dictionary
from `ConsMarkovModel`, which extends the standard incomplete-markets
consumption-saving setup with a symmetric 2-state Markov chain
(p11 = p22 = 0.9). State-dependent parameters (Rfree, LivPrb, PermGroFac)
are set identical across states so the only variation comes from the Markov
transition itself—an intentional simplification for validation purposes.

In [2]:
params = deepcopy(init_indshk_markov)
params["Mrkv_p11"] = [0.9]
params["Mrkv_p22"] = [0.9]
params["Rfree"] = [np.array([1.03, 1.03])]
params["LivPrb"] = [np.array([0.98, 0.98])]
params["PermGroFac"] = [np.array([1.0, 1.0])]
params["cycles"] = 0

agent = MarkovConsumerType(**params)
print("MrkvArray:", agent.MrkvArray[0])

MrkvArray: [[0.9 0.1]
 [0.1 0.9]]


## 2. Compute PE steady state using the new method

In [3]:
A_ss, C_ss = agent.compute_pe_steady_state()
print(f"A_ss = {A_ss:.6f}")
print(f"C_ss = {C_ss:.6f}")

A_ss = 0.835777
C_ss = 1.007856


## 3. Validate transition matrix column sums

In [4]:
col_sums = agent.tran_matrix.sum(axis=0)
print(f"Column sums: min={col_sums.min():.12f}, max={col_sums.max():.12f}")
assert np.allclose(col_sums, 1.0, atol=1e-10), "Column sums are not 1.0!"
print("PASS: All column sums equal 1.0")

Column sums: min=1.000000000000, max=1.000000000000
PASS: All column sums equal 1.0


## 4. Validate ergodic Markov state fractions

In [5]:
M = len(agent.dist_mGrid)  # number of grid points in the asset distribution
J = agent.MrkvArray[0].shape[0]  # number of Markov states

# vec_erg_dstn is a single vector of length M*J, stacked [state0, state1, ...].
# Sum each state's block to get the marginal probability of being in that state.
pi_by_state = [np.sum(agent.vec_erg_dstn[j * M : (j + 1) * M]) for j in range(J)]
print(f"Ergodic state fractions: {pi_by_state}")
print("Expected (symmetric p11=p22=0.9): [0.5, 0.5]")
assert abs(pi_by_state[0] - 0.5) < 0.01, "State 0 fraction should be ~0.5"
assert abs(pi_by_state[1] - 0.5) < 0.01, "State 1 fraction should be ~0.5"
print("PASS: Ergodic Markov fractions match analytical values")

Ergodic state fractions: [np.float64(0.5), np.float64(0.5)]
Expected (symmetric p11=p22=0.9): [0.5, 0.5]
PASS: Ergodic Markov fractions match analytical values


## 5. Compare with hand-built TM

In [6]:
from HARK.utilities import gen_tran_matrix_1D_markov, jump_to_grid_1D

# Rebuild the TM by hand using the same ingredients the method uses internally,
# so any difference would indicate a bug in the method's assembly logic.
dist_mGrid = agent.dist_mGrid
MrkvArr = agent.MrkvArray[0]
Rfree_arr = np.asarray(agent.Rfree[0], dtype=np.float64)
PermGroFac_arr = np.asarray(agent.PermGroFac[0], dtype=np.float64)
LivPrb_arr = np.asarray(agent.LivPrb[0], dtype=np.float64)

# aPol_Grid is stored per-state; stack into (J, M) array for the utility function
aPol_2d = np.array([agent.aPol_Grid[j] for j in range(J)])

# Unpack the joint income-shock distribution for period 0, state 0
# (all states share the same shock distribution in this symmetric calibration)
shk_prbs = agent.IncShkDstn[0][0].pmv
perm_shks = agent.IncShkDstn[0][0].atoms[0]
tran_shks = agent.IncShkDstn[0][0].atoms[1]

# Newborn distribution: agents who die are replaced at m = 1 (normalized),
# spread across the asset grid according to transitory-shock realizations
newborn_1d = jump_to_grid_1D(np.ones_like(tran_shks), shk_prbs, dist_mGrid)
markov_stationary = MarkovConsumerType._calc_markov_stationary(MrkvArr)
NewBornDist = np.zeros(M * J)
for jp in range(J):
    # Weight each state's newborn mass by the Markov stationary probability
    NewBornDist[jp * M : (jp + 1) * M] = markov_stationary[jp] * newborn_1d

hand_tm = gen_tran_matrix_1D_markov(
    dist_mGrid,
    aPol_2d,
    MrkvArr,
    Rfree_arr,
    PermGroFac_arr,
    LivPrb_arr,
    shk_prbs,
    perm_shks,
    tran_shks,
    NewBornDist,
)

max_diff = np.max(np.abs(hand_tm - agent.tran_matrix))
print(f"Max element-wise difference between method TM and hand-built TM: {max_diff}")
assert max_diff < 1e-14, f"TMs should be identical, got diff={max_diff}"
print("PASS: Method TM matches hand-built TM exactly")

Max element-wise difference between method TM and hand-built TM: 0.0
PASS: Method TM matches hand-built TM exactly


## 6. Summary

All validations passed:

| Check | Result |
|-------|--------|
| Column sums = 1.0 | PASS |
| Ergodic Markov fractions match analytical | PASS |
| `compute_pe_steady_state()` returns finite positive values | PASS |
| Method TM = hand-built TM | PASS |

The new `MarkovConsumerType` TM methods are validated and ready for use.
